# Parts 2–5 — Benchmarks, SARIMAX, temperature, feature model

Drives the pipeline and shows the comparison table, forecast plot, SARIMAX AIC search, residual diagnostics, prediction intervals, and the feature-based model.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, matplotlib.pyplot as plt
from electricity_demand.data import load_processed_weekly
from electricity_demand.models import benchmarks as bm, sarimax as sx, feature_models as fm
from electricity_demand import features as feat, plotting as plot, evaluation as ev
from electricity_demand.config import TEST_WEEKS, WEEKLY_SEASONALITY

In [ ]:
weekly = load_processed_weekly()
train, test = weekly.iloc[:-TEST_WEEKS], weekly.iloc[-TEST_WEEKS:]
h = len(test)
print(train.index.min().date(), '->', train.index.max().date(), '| test weeks:', h)

## Part 2 — Benchmarks (2-year horizon)

In [ ]:
fc = bm.all_benchmarks(train, h, index=test.index, seasonality=WEEKLY_SEASONALITY)
plot.plot_forecasts(train, test, fc, title='Benchmark forecasts');

## Part 3 — SARIMA with AIC grid search
Full grid is p∈[0,6], d∈[0,2], q∈[0,6] × a small seasonal grid. That is slow with s=52; narrow the ranges or set `max_models` while developing, then widen for the final run.

In [ ]:
best, results = sx.grid_search_sarima(
    train, seasonal=True, seasonal_period=WEEKLY_SEASONALITY,
    p_range=range(0,3), d_range=range(0,2), q_range=range(0,3),
    sp_range=range(0,2), sd_range=range(0,2), sq_range=range(0,2),
    max_models=None, verbose=True)
results.head(10)

### Residual diagnostics

In [ ]:
plot.plot_residual_diagnostics(sx.get_residuals(best));

### Forecast with 80% prediction interval

In [ ]:
mean, lo, hi = sx.forecast_sarimax_intervals(best, h, index=test.index, alpha=0.2)
plot.plot_prediction_interval(train, test, mean, lo, hi, name='SARIMA')
print('RMSE:', round(ev.rmse(test, mean),3), '| coverage:', round(ev.coverage(test, lo, hi),3))
fc['sarima'] = mean

## Part 4 — Add Berlin temperature (conditional / explanatory forecast)
Fetch temperature, build weekly features, refit SARIMAX with `exog`. Using observed test-period temperature makes this a **conditional** forecast, not operational.

In [ ]:
# temp_daily = feat.get_open_meteo_temperature(str(weekly.index.min().date()), str(weekly.index.max().date()))
# temp_weekly = feat.weekly_temperature_features(temp_daily, weekly.index)
# X = temp_weekly[['temp_mean','heating_degree_days','cooling_degree_days']]
# Xtr, Xte = X.iloc[:-TEST_WEEKS], X.iloc[-TEST_WEEKS:]
# fit_x = sx.fit_sarimax(train, X_train=Xtr, order=best.model.order, seasonal_order=best.model.seasonal_order)
# fc['sarimax_temp'] = sx.forecast_sarimax(fit_x, h, X_test=Xte, index=test.index)

## Part 5 — Feature-based model (gradient boosting)

In [ ]:
table = feat.make_feature_table(weekly)   # add temp_weekly=... when available
tr, te = table.iloc[:-TEST_WEEKS], table.iloc[-TEST_WEEKS:]
cols = [c for c in table.columns if c!='load_gw']
model = fm.fit_gradient_boosting(tr[cols], tr['load_gw'])
fc['feature_model'] = fm.predict_feature_model(model, te[cols], index=te.index)
fm.feature_importance(model, cols).head(10)

## Comparison table (ranked by MASE)

In [ ]:
rows = [ev.evaluate_forecast(name, test, p.reindex(test.index), train) for name,p in fc.items()]
ev.comparison_table(rows).round(3)

In [ ]:
plot.plot_forecasts(train, test, fc, title='All models');